# Position-first candidate attraction: complete prototype

This notebook implements the experiment fixed in `SIGN_ALIGNMENT_3_CANDIDATE_OPTIMIZATION.md` and reproduces the `NBC.4020`, crop 3 setup from `signs_alignment.ipynb`. It first runs and visualizes the established pipeline through `align_text_rows`, including the original `alignment_diagnostic`. It then runs the isolated fixed-candidate optimization and produces its own complete diagnostic chain.

The experimental code lives only in `pipeline_3_candidate_test.py`. Original alignment fields are not overwritten; experimental outputs use the `candidate_test_*` namespace and a separate output directory. Visualization palette: green is reserved for GT; anchors are purple, fixed candidates red, movable text blue, final candidate matches yellow, and NULL cyan.

In [ ]:
%reload_ext autoreload
%autoreload 2
%cd ~/erc-src/cuneiform-ocr-sign-alignment-worktree
%env PATH=$HOME/.local/bin:$PATH

import os
from dotenv import load_dotenv

from sign_alignment.detector import ModelConfig, TabletImageDetector
from sign_alignment.data_source import LocalDataSource
from sign_alignment.visualizer import ColorConfig
from sign_alignment.pipeline import CropContext, Runner, VisOptions
import sign_alignment.pipeline as pp
import pipeline_3_candidate_test as pp3

ANNOTATIONS_DIR = os.path.expanduser("~/erc-work-data/data-of-cuneiform-ocr-data/filtered_annotations")
CONFIG_FILE = "configs/detr.py"
CHECKPOINT_FILE = os.path.expanduser("~/erc-work-data/retrained_models/detr-173/epoch_1000.pth")
SCORE_THRESHOLD = 0.0
OUTPUT_DIR = "alignment_results_3_candidate_test"
CROP_INDEX = 3    # same crop as signs_alignment.ipynb
load_dotenv()

In [ ]:
model_config = ModelConfig(
    config_file=CONFIG_FILE,
    checkpoint_file=CHECKPOINT_FILE,
    device="auto",
)
if "tablet_detector" not in globals() or getattr(tablet_detector, "model", None) is None:
    tablet_detector = TabletImageDetector(
        default_score_threshold=SCORE_THRESHOLD,
        model_config=model_config,
        keep_crops=True,
        is_crop_itself=False,
    )
else:
    print("Reusing existing tablet_detector instance.")

crop_context = CropContext(
    tablet_detector=tablet_detector,
    local_source=LocalDataSource(ANNOTATIONS_DIR),
    color_config=ColorConfig,
    output_dir=OUTPUT_DIR,
    img_idx=CROP_INDEX,
    task_type="candidate_test",
    gt_visualization_excluded_prefixes=("SURFACE_",),
)
vis = VisOptions(info=True, display=True, save=True)
runner = Runner(context=crop_context, vis=vis)

## A. Established pipeline and coarse-alignment diagnostics

Every stage is kept in a separate cell so its state and visualization can be inspected independently.

In [ ]:
runner.choose_sample(name="CBS.1499")
runner.choose_sample(name="BM.34110")
runner.choose_sample(name="CBS.1516")
runner.choose_sample(name="YBC.7794")
# runner.choose_sample(name="HS.2020")
# runner.choose_sample(name="BM.40807")
runner.run([pp.Step("Load data", pp.load_data, pp.vis_loaded_data)])

In [ ]:
runner.choose_crop(2)
runner.run([pp.Step("Detect signs", pp.detect_signs, pp.vis_detections)])

In [ ]:
runner.run([pp.Step(
    "Transform GT to crop", pp.transform_gt_to_crop, pp.vis_crop_ground_truth
)])

In [ ]:
runner.run([pp.Step(
    "Detection statistics", lambda _: None, pp.vis_detection_statistics
)])

In [ ]:
runner.run([pp.Step("Create box sets", pp.create_box_sets, pp.vis_box_sets)])

In [ ]:
runner.run([pp.Step("Detect rows", pp.detect_rows, pp.vis_detected_rows_info)])

In [ ]:
runner.run([pp.Step("Match rows", pp.match_rows, pp.vis_row_matches)])

In [ ]:
# Detection rows, D# -> R# mapping, and the Hough parameter space.
runner.run([pp.Step(
    "Visualize detection rows", lambda _: None, pp.vis_detection_rows
)])

In [ ]:
runner.run([pp.Step(
    "Match signs in rows", pp.match_signs_in_rows, pp.vis_sign_matches
)])

In [ ]:
runner.run([pp.Step(
    "Align text rows", pp.align_text_rows, pp.vis_aligned_rows
)])

In [ ]:
# Original text mapping, rows side-by-side, and the important coarse alignment_diagnostic.
runner.run([pp.Step(
    "Build original sign match info", pp.build_sign_match_info, pp.vis_sign_match_info
)])

In [ ]:
runner.run([pp.Step(
    "Coarse offset analysis", lambda _: None, pp.vis_offset_analysis
)])

In [ ]:
# Existing detection-geometry/relabel baseline, kept for comparison.
runner.run([pp.Step(
    "Result without optimization",
    pp.create_result_without_optimization,
    pp.vis_result_without_optimization,
)])

In [ ]:
# Candidate optimization only needs stored boxes; free detector VRAM first.
runner.run([pp.Step("Unload detector", pp.unload_detector)])

## B. Fixed-candidate optimization

The first annealing stage is pure position. Later stages add candidate size, weak objectness and bounded positive class/diff bonuses. Detector mismatch never adds a penalty. Candidate columns have capacity one, while NULL has unlimited capacity. Final candidate selection uses an ordered partial assignment with one-to-one candidate capacity.

In [ ]:
candidate_config = pp3.CandidateAttractionConfig(
    temperatures=(2.0, 1.0, 0.5, 0.25),
    steps_per_temperature=35,
    learning_rate=0.04,
    class_bonus=0.06,       # bounded tie-breaker only
    objectness_bonus=0.10,  # only used after pure-position capture
    device="auto",
)
runner.run([pp.Step(
    "Run candidate attraction",
    lambda context: pp3.run_candidate_attraction(context, candidate_config),
)])

In [ ]:
# Raw detections grouped into fixed physical candidates; anchors are already consumed.
runner.run([pp.Step(
    "Visualize candidate pool", lambda _: None, pp3.vis_candidate_pool
)])

In [ ]:
# Continuous text positions after every temperature stage, followed by final snapping.
runner.run([pp.Step(
    "Annealing snapshots", lambda _: None, pp3.vis_candidate_annealing_snapshots
)])

In [ ]:
# Per-row final soft probabilities, pair costs, gated edges, and hard DP choice.
runner.run([pp.Step(
    "Assignment matrices",
    lambda _: None,
    lambda context, vis: pp3.vis_candidate_assignment_matrices(
        context, VisOptions(info=vis.info, display=vis.display, save=False)
    ),
)])

In [ ]:
# Fixed candidates, initial centers, movement paths, and final status-colored boxes.
runner.run([pp.Step(
    "Candidate attraction overlay", lambda _: None, pp3.vis_candidate_attraction
)])

In [ ]:
runner.run([pp.Step(
    "Annealing history", lambda _: None, pp3.vis_candidate_attraction_history
)])

In [ ]:
# Candidate-specific match info and alignment_diagnostic, plus before/after comparison.
runner.run([pp.Step(
    "Candidate alignment diagnostic",
    pp3.build_candidate_sign_match_info,
    pp3.vis_candidate_alignment_diagnostic,
)])

In [ ]:
# 2x2: coarse, final, detection overlay, and GT overlay.
runner.run([pp.Step(
    "Candidate results comparison", lambda _: None, pp3.vis_candidate_results_comparison
)])

In [ ]:
# Center/size changes and assignment counts, split by anchor/candidate/NULL.
runner.run([pp.Step(
    "Candidate parameter changes", lambda _: None, pp3.vis_candidate_parameter_changes
)])

## C. Inspect assignments and isolation

In [ ]:
records = pp3.candidate_attraction_records(runner.context)
try:
    import pandas as pd
    candidate_table = pd.DataFrame(records)
    display(candidate_table[[
        "text_row_idx", "text_idx", "sign_name", "input_status",
        "output_status", "candidate_idx", "detector_labels",
        "class_support", "soft_probability", "null_probability",
        "included_in_result", "movement_px",
    ]])
except ImportError:
    candidate_table = records
    candidate_table[:20]

In [ ]:
# These are the intended cases where geometry wins despite missing class support.
position_only = [
    row for row in records
    if row["output_status"] == "candidate" and row["class_support"] <= 0.0
]
print(f"Position-only/wrong-label candidate matches: {len(position_only)}")
position_only[:20]

In [ ]:
state = runner.context.state
print("original aligned_boxes:", len(state.aligned_boxes))
print("candidate_test_boxes:  ", len(state.candidate_test_boxes))
print("separate box objects:   ", state.aligned_boxes is not state.candidate_test_boxes)
print("original final_boxes:   ", state.final_boxes)
print("experimental fields: candidate_test_boxes, candidate_test_rows, candidate_test_run")